In [1]:
import argparse
import json
import os
import shutil
import subprocess
from concurrent.futures import ProcessPoolExecutor
from fractions import Fraction
from pathlib import Path
from tempfile import TemporaryDirectory

In [2]:
def run(cmd):
    return subprocess.run(cmd, check=True, capture_output=True, text=True).stdout

In [3]:
def probe(path):
    try:
        data = json.loads(run([
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries", "stream=r_frame_rate,avg_frame_rate",
            "-of", "json", str(path),
        ]))["streams"][0]
        rate = Fraction(data["r_frame_rate"])
        print(f'r-frame: {data["r_frame_rate"]}, avg- {data['avg_frame_rate']}')
        if rate > 0 and abs(float(rate) - float(Fraction(data["avg_frame_rate"]))) < 10.0:
            return path, float(Fraction(data["avg_frame_rate"]))
        print(f"SKIP {path.name}: unequal/invalid FPS", flush=True)
    except (subprocess.CalledProcessError, ValueError, ZeroDivisionError,
            KeyError, IndexError) as exc:
        print(f"SKIP {path.name}: {getattr(exc, 'stderr', None) or exc}", flush=True)
    return path, None

In [4]:
abs(1-2)

1

In [5]:
def extract(job):
    path, rate, output = job

    # Lấy frame 0, 40, 80, 120, ...
    expr = "eq(mod(n,40),0)"

    vf = (f"[0:v:0]select='{expr}',split=2[a][b];"
          "[a]scale=720:-2[full];[b]scale=192:-2[thumb]")

    target = output / path.stem

    try:
        with TemporaryDirectory(prefix=".frames-", dir=output) as tmp:
            tmp = Path(tmp)

            cmd = [
                "ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin",
                "-y", "-threads", "1", "-i", str(path),
                "-filter_complex_threads", "1",
                "-filter_complex", vf
            ]

            for label, ext, codec in (
                ("full", "jpg", ["-c:v", "mjpeg", "-q:v", "3"]),
                ("thumb", "webp", ["-c:v", "libwebp", "-quality", "80"]),
            ):
                cmd += [
                    "-map", f"[{label}]",
                    "-vsync", "0",
                    *codec,
                    "-threads", "1",
                    "-start_number", "0",
                    str(tmp / f"%09d.{ext}")
                ]

            run(cmd)

            jpgs = sorted(tmp.glob("*.jpg"))
            webps = sorted(tmp.glob("*.webp"))

            if not jpgs or len(jpgs) != len(webps):
                raise RuntimeError("Missing or mismatched output frames")

            for folder in ("selected_frame", "thumbnails"):
                (tmp / folder).mkdir()

            for jpg in jpgs:
                k = int(jpg.stem)

                # Vì k là index của frame đã chọn:
                # k=0 -> frame gốc 0
                # k=1 -> frame gốc 40
                # k=2 -> frame gốc 80
                fid = k * 40

                for ext, folder in (
                    ("jpg", "selected_frame"),
                    ("webp", "thumbnails")
                ):
                    jpg.with_suffix(f".{ext}").rename(
                        tmp / folder / f"{path.stem}-{fid:05d}.{ext}"
                    )

            target.mkdir(exist_ok=False)

            for folder in ("selected_frame", "thumbnails"):
                shutil.move(str(tmp / folder), str(target / folder))

        print(f"OK {path.stem}: {len(jpgs)} frames", flush=True)
        return True

    except Exception as exc:
        print(
            f"ERROR {path.stem}: {getattr(exc, 'stderr', None) or exc}",
            flush=True
        )
        return False

In [6]:
def main(input_path, output_path, workers=4):
    
  
    if not input_path.is_dir() or workers < 1:
        print("input_path must be a directory; workers must be >= 1")
        return
        
    for tool in ("ffmpeg", "ffprobe"):
        if not shutil.which(tool):
            print(f"Missing {tool}")
            return
            
    videos = sorted(p for p in input_path.iterdir()
                    if p.is_file() and p.suffix.lower() == ".mov")
    if not videos:
        print("No .mov videos found")
        return
        
    if len({p.stem for p in videos}) != len(videos):
        print("Duplicate video IDs")
        return
        
    output = output_path
    output.mkdir(parents=True, exist_ok=True)
    
    if (output / "fps.json").exists() or any((output / p.stem).exists() for p in videos):
        print("Use a fresh output directory; matching outputs already exist")
        return
        
    with ProcessPoolExecutor(max_workers=min(workers, len(videos))) as pool:
        rates = [(path, rate) for path, rate in pool.map(probe, videos) if rate is not None]
        fps = {path.stem: float(rate) for path, rate in rates}
        (output / "fps.json").write_text(json.dumps(fps, indent=2), encoding="utf-8")
        results = list(pool.map(extract, ((p, r, output) for p, r in rates)))
        
    print(f"Done: {sum(results)}/{len(rates)} videos; FPS: {output / 'fps.json'}")
    
    if not all(results):
        raise SystemExit(1)

In [7]:
INPUT_PATH = Path("/kaggle/input/datasets/nhnguynvn1902/videon011-n020/videos")
OUTPUT_PATH = Path("/kaggle/working")

In [8]:
main(INPUT_PATH, OUTPUT_PATH)

r-frame: 25/1, avg- 1500100/60001r-frame: 30/1, avg- 3558200/118603r-frame: 30/1, avg- 909000/30299r-frame: 30/1, avg- 4575250/152503



r-frame: 25/1, avg- 15939200/639921
r-frame: 25/1, avg- 47875/1923
r-frame: 25/1, avg- 625000/24999r-frame: 25/1, avg- 7500500/300007

r-frame: 25/1, avg- 48000000/1919891
r-frame: 25/1, avg- 48003200/1920019
r-frame: 25/1, avg- 2492500/100003
r-frame: 25/1, avg- 192012800/7680089
r-frame: 30/1, avg- 138086400/4608161
r-frame: 30/1, avg- 9198592/307179r-frame: 30/1, avg- 27592704/921643

r-frame: 25/1, avg- 14925000/596969
r-frame: 25/1, avg- 1865750/74627
r-frame: 25/1, avg- 2985200/119403
r-frame: 25/1, avg- 2985000/119393
r-frame: 25/1, avg- 4975000/198989
r-frame: 25/1, avg- 7426500/297043
r-frame: 25/1, avg- 7500000/299987
r-frame: 25/1, avg- 2995600/119817
r-frame: 25/1, avg- 7537500/301483
r-frame: 25/1, avg- 7500000/299989
r-frame: 25/1, avg- 5000000/200001
r-frame: 25/1, avg- 7496500/300009
r-frame: 25/1, avg- 7500500/300001
r-frame: 25/1, av

In [9]:
!zip -r /kaggle/working/output.zip /kaggle/working

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/N020-V003/ (stored 0%)
  adding: kaggle/working/N020-V003/selected_frame/ (stored 0%)
  adding: kaggle/working/N020-V003/selected_frame/N020-V003-13280.jpg (deflated 0%)
  adding: kaggle/working/N020-V003/selected_frame/N020-V003-14880.jpg (deflated 0%)
  adding: kaggle/working/N020-V003/selected_frame/N020-V003-14400.jpg (deflated 0%)
  adding: kaggle/working/N020-V003/selected_frame/N020-V003-00560.jpg (deflated 0%)
  adding: kaggle/working/N020-V003/selected_frame/N020-V003-02160.jpg (deflated 0%)
  adding: kaggle/working/N020-V003/selected_frame/N020-V003-12360.jpg (deflated 0%)
  adding: kaggle/working/N020-V003/selected_frame/N020-V003-10240.jpg (deflated 0%)
  adding: kaggle/working/N020-V003/selected_frame/N020-V003-09600.jpg (deflated 0%)
  adding: kaggle/working/N020-V003/selected_frame/N020-V003-12920.jpg (deflated 0%)
  adding: kaggle/working/N020-V003/selected_frame/N020-V003-12480.jpg (deflated 0%)
  adding: k

In [10]:
import os
os.remove('/kaggle/working/fps.json')